In [1]:
import torch
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

c:\Users\VISHNU\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

In [ ]:
MODEL_NAME = "Qwen/Qwen3-1.7B"

print("CUDA available:", torch.cuda.is_available())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
).to("cuda")

model.eval()

print("Qwen3 loaded successfully")
print("Model device:", next(model.parameters()).device)

CUDA available: True


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Qwen3 loaded successfully
Model device: cuda:0


In [ ]:
def ask_qwen(prompt, max_new_tokens=500):
    messages = [
        {
            "role": "system",
            "content": "You are a query understanding module. Return only valid JSON."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
def parse_json_response(response):
    response = response.strip()

    response = re.sub(r"```json\s*", "", response)
    response = re.sub(r"```\s*", "", response)

    start = response.find("{")
    end = response.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No JSON object found in Qwen response")

    response = response[start:end + 1]

    return json.loads(response)

In [ ]:
NEGATION_TRIGGERS = [
    "did not",
    "does not",
    "do not",
    "didn't",
    "doesn't",
    "don't",
    "without",
    "never",
    "but not",
    "excluding",
    "exclude",
    "not",
    "no"
]

In [ ]:
def find_negation_triggers(query):
    query_lower = query.lower()
    found = []

    for trigger in NEGATION_TRIGGERS:
        if trigger in query_lower:
            found.append(trigger)

    return found

In [ ]:
def verify_negation(query, structured_result):
    detected_triggers = find_negation_triggers(query)

    negation_events = {
        item.get("event_id"): item.get("trigger")
        for item in structured_result.get("negations", [])
    }

    verification_results = []

    for event in structured_result.get("events", []):
        event_id = event.get("event_id")
        qwen_negated = event.get("negated", False)

        expected_negated = event_id in negation_events
        trigger = negation_events.get(event_id)

        if qwen_negated == expected_negated:
            status = "VALID"
        else:
            status = "CORRECTED"

        if expected_negated:
            event["negated"] = True

        verification_results.append({
            "event_id": event_id,
            "action": event.get("action"),
            "original_negated": qwen_negated,
            "verified_negated": event.get("negated", False),
            "negation_trigger": trigger,
            "verification": status
        })

    if detected_triggers and not negation_events:
        verification_results.append({
            "verification": "REVIEW",
            "detected_triggers": detected_triggers,
            "message": "Negation detected in query but Qwen did not assign it to an event."
        })

    return verification_results

In [ ]:
VEHICLE_KEYWORDS = {
    "car",
    "truck",
    "bus",
    "van",
    "motorcycle",
    "motorbike",
    "bicycle",
    "bike",
    "scooter",
    "vehicle"
}

TEMPORAL_TRIGGERS = [
    "then",
    "after",
    "before",
    "later",
    "followed by",
    "next",
    "finally",
    "first"
]

def verify_entities(query, structured_result):
    entities = structured_result.get("entities", [])
    verified_entities = []
    seen = set()

    for entity in entities:
        name = str(entity.get("name", "")).strip()
        entity_type = entity.get("type", "").strip().lower()

        if not name:
            continue

        key = name.lower()

        if key in seen:
            continue

        seen.add(key)

        words = key.split()

        if any(vehicle in words for vehicle in VEHICLE_KEYWORDS):
            entity_type = "vehicle"

        verified_entities.append({
            "type": entity_type,
            "name": name
        })

    structured_result["entities"] = verified_entities

    return structured_result

In [ ]:
TEMPORAL_PATTERNS = [
    "followed by",
    "after that",
    "and then",
    "then",
    "later",
    "after",
    "before",
    "next",
    "finally",
    "first"
]

def verify_temporal_relations(query, structured_result):
    events = structured_result.get("events", [])

    if len(events) < 2:
        structured_result["temporal_relations"] = []
        return structured_result

    query_lower = query.lower()

    relations = []

    def add_relation(event1, event2):
        if event1 == event2:
            return

        relation = {
            "relation": "before",
            "event1": event1,
            "event2": event2
        }

        if relation not in relations:
            relations.append(relation)

    event_ids = [event["event_id"] for event in events]

    has_temporal_marker = any(
        pattern in query_lower
        for pattern in TEMPORAL_PATTERNS
    )

    if not has_temporal_marker:
        structured_result["temporal_relations"] = []
        return structured_result

    if "first" in query_lower:
        first_event = events[0]

        if len(events) > 1:
            add_relation(
                first_event["event_id"],
                events[1]["event_id"]
            )

    if "then" in query_lower or "and then" in query_lower:
        then_position = query_lower.find("then")

        before_then = []
        after_then = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < then_position:
                before_then.append(event)
            else:
                after_then.append(event)

        if before_then and after_then:
            add_relation(
                before_then[-1]["event_id"],
                after_then[0]["event_id"]
            )

    if "later" in query_lower:
        later_position = query_lower.find("later")

        before_later = []
        after_later = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < later_position:
                before_later.append(event)
            elif position > later_position:
                after_later.append(event)

        if before_later and after_later:
            add_relation(
                before_later[-1]["event_id"],
                after_later[0]["event_id"]
            )

    if "followed by" in query_lower:
        followed_position = query_lower.find("followed by")

        before_followed = []
        after_followed = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < followed_position:
                before_followed.append(event)
            elif position > followed_position:
                after_followed.append(event)

        if before_followed and after_followed:
            add_relation(
                before_followed[-1]["event_id"],
                after_followed[0]["event_id"]
            )

    if "after" in query_lower:
        after_position = query_lower.find("after")

        before_after = []
        after_after = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < after_position:
                before_after.append(event)
            elif position > after_position:
                after_after.append(event)

        if before_after and after_after:
            add_relation(
                before_after[-1]["event_id"],
                after_after[0]["event_id"]
            )

    if "before" in query_lower:
        before_position = query_lower.find("before")

        before_events = []
        after_events = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < before_position:
                before_events.append(event)
            elif position > before_position:
                after_events.append(event)

        if before_events and after_events:
            add_relation(
                after_events[0]["event_id"],
                before_events[-1]["event_id"]
            )

    if "next" in query_lower:
        next_position = query_lower.find("next")

        before_next = []
        after_next = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < next_position:
                before_next.append(event)
            elif position > next_position:
                after_next.append(event)

        if before_next and after_next:
            add_relation(
                before_next[-1]["event_id"],
                after_next[0]["event_id"]
            )

    if "finally" in query_lower:
        finally_position = query_lower.find("finally")

        before_finally = []
        after_finally = []

        for event in events:
            action = event.get("action", "").lower()

            if not action:
                continue

            position = query_lower.find(action)

            if position == -1:
                continue

            if position < finally_position:
                before_finally.append(event)
            elif position > finally_position:
                after_finally.append(event)

        if before_finally and after_finally:
            add_relation(
                before_finally[-1]["event_id"],
                after_finally[0]["event_id"]
            )

    if "first" in query_lower and "then" in query_lower:
        for i in range(len(event_ids) - 1):
            add_relation(
                event_ids[i],
                event_ids[i + 1]
            )

    if "then" in query_lower and len(event_ids) > 2:
        for i in range(len(event_ids) - 1):
            add_relation(
                event_ids[i],
                event_ids[i + 1]
            )

    if "followed by" in query_lower and len(event_ids) > 2:
        for i in range(len(event_ids) - 1):
            add_relation(
                event_ids[i],
                event_ids[i + 1]
            )

    if "finally" in query_lower and len(event_ids) > 2:
        for i in range(len(event_ids) - 1):
            add_relation(
                event_ids[i],
                event_ids[i + 1]
            )

    structured_result["temporal_relations"] = relations

    return structured_result

In [ ]:
PROMPT_FINAL = """
You are the Query Understanding module of a video semantic search system.

Convert the user's natural-language video search query into ONE structured JSON object for a visual retrieval system.

The goal is to preserve all explicitly stated information while keeping entities, attributes, events, relationships, temporal order, and negations clearly separated.

CORE PRINCIPLE:

ENTITY = the actual person, object, vehicle, clothing item, or location.

ATTRIBUTE = a word or phrase that describes an entity.

Always separate the main entity from its descriptive words.

Examples:
"red top hat"
-> entity: hat
-> attributes: ["red", "top"]

"black leather jacket"
-> entity: jacket
-> attributes: ["black", "leather"]

"small blue bag"
-> entity: bag
-> attributes: ["small", "blue"]

"white car"
-> entity: car
-> attributes: ["white"]

Never use "red top hat" as the entity name when "hat" is the actual entity.

==================================================
1. ENTITY EXTRACTION
==================================================

Extract every explicitly mentioned:

- person
- object
- vehicle
- clothing item
- location

Each entity must appear exactly once in the "entities" array.

Use the most specific entity type:

person:
- man
- woman
- boy
- girl
- person
- child

vehicle:
- car
- truck
- bus
- van
- motorcycle
- bicycle

clothing:
- hat
- shirt
- jacket
- coat
- dress
- pants
- trousers
- shoes
- etc.

object:
- bag
- phone
- bottle
- ball
- umbrella
- etc.

location:
- road
- street
- building
- park
- room
- shop
- etc.

==================================================
2. ENTITY NAMES
==================================================

The "name" must contain ONLY the core entity.

Remove descriptive adjectives from the entity name.

Examples:

"red hat"
-> name: "hat"

"red top hat"
-> name: "hat"

"black leather jacket"
-> name: "jacket"

"large black bag"
-> name: "bag"

"white sedan"
-> name: "sedan"

"small white car"
-> name: "car"

Do NOT create:

"name": "red hat"

Do NOT create:

"name": "red top hat"

The descriptive words belong in attributes.

==================================================
3. ATTRIBUTE EXTRACTION
==================================================

Extract every explicitly stated adjective or descriptive phrase that describes an entity.

Attributes must always be represented as a JSON array of strings.

Examples:

"red top hat"
-> entity:
{
    "type": "clothing",
    "name": "hat"
}
-> attributes:
["red", "top"]

"black leather jacket"
-> entity:
{
    "type": "clothing",
    "name": "jacket"
}
-> attributes:
["black", "leather"]

"large red bag"
-> entity:
{
    "type": "object",
    "name": "bag"
}
-> attributes:
["large", "red"]

"white car"
-> entity:
{
    "type": "vehicle",
    "name": "car"
}
-> attributes:
["white"]

Preserve ALL explicitly stated attributes.

Do not discard adjectives.

Do not invent attributes.

Do not infer attributes.

==================================================
4. ATTRIBUTE OWNERSHIP
==================================================

Every attribute must belong to the entity it explicitly describes.

Example:

"A man wearing a red top hat entered a white car."

Entities:

man
hat
car

Attributes:

hat -> ["red", "top"]
car -> ["white"]

The event should preserve these attributes when the entity participates in the event.

==================================================
5. PEOPLE
==================================================

A person can have attributes.

Example:

"A tall man wearing a black coat entered a car."

Person:
{
    "type": "person",
    "name": "man"
}

Person attributes:
["tall"]

Clothing:
{
    "type": "clothing",
    "name": "coat"
}

Clothing attributes:
["black"]

The clothing itself is an entity, but "wearing" does NOT automatically become a separate event.

==================================================
6. CLOTHING
==================================================

When clothing describes a person performing an action, do NOT create a separate wearing event.

Example:

"A man wearing a black jacket entered a car."

Correct:

Entity:
man

Entity:
jacket
attribute: black

Event:
man entered car

The jacket can be represented through the event's attributes or relations.

Do NOT create:

Event:
man wearing jacket

unless the query explicitly treats the wearing action itself as an event.

==================================================
7. OBJECTS
==================================================

Objects must be separated from their descriptive attributes.

Example:

"red ball"

Correct:
entity = ball
attributes = ["red"]

Example:

"small black bag"

Correct:
entity = bag
attributes = ["small", "black"]

Never merge adjectives into the entity name.

==================================================
8. VEHICLES
==================================================

Vehicles must use the entity type "vehicle".

Examples:

"white car"
-> type: vehicle
-> name: car
-> attributes: ["white"]

"large blue truck"
-> type: vehicle
-> name: truck
-> attributes: ["large", "blue"]

"black motorcycle"
-> type: vehicle
-> name: motorcycle
-> attributes: ["black"]

==================================================
9. EVENTS
==================================================

Separate distinct actions into separate events.

Every event MUST use exactly this structure:

{
    "event_id": 1,
    "action": "...",
    "subject": "...",
    "object": "...",
    "attributes": [],
    "relations": [],
    "negated": false
}

Use the entity's CORE NAME in subject and object.

Example:

"A man wearing a red hat entered a white car."

Event:

{
    "event_id": 1,
    "action": "entering",
    "subject": "man",
    "object": "car",
    "attributes": ["red hat", "white"],
    "relations": [],
    "negated": false
}

Do not invent actions.

==================================================
10. SHARED SUBJECTS
==================================================

If multiple events refer to the same person, use the same subject.

Example:

"A man entered a car and then walked away."

Events:

Event 1:
subject = man
action = entering
object = car

Event 2:
subject = man
action = walking
object = null

==================================================
11. RELATIONSHIPS
==================================================

Relationships describe how entities are connected.

Examples:

wearing
carrying
holding
standing near
behind
in front of
next to
inside
near
with

Use relations when the relationship is explicitly stated.

Example:

"A man carrying a bag walked."

The relationship can be represented as:

"relations": [
    {
        "relation": "carrying",
        "subject": "man",
        "object": "bag"
    }
]

Do not invent relationships.

==================================================
12. TEMPORAL ORDER
==================================================

Preserve the order of explicitly sequential events.

Temporal indicators include:

then
after
before
followed by
next
first
later
finally

If two events are explicitly sequential, create a temporal relation.

Example:

"A woman picked up a bag and then entered a car."

Events:

1 = picked up bag
2 = entered car

Temporal relation:

{
    "relation": "before",
    "event1": 1,
    "event2": 2
}

==================================================
13. TEMPORAL RULES
==================================================

Only create temporal relationships when the query explicitly indicates temporal order.

Do NOT assume temporal order merely because one event appears before another in the sentence.

"but", "however", and "although" are NOT temporal indicators.

==================================================
14. NEGATION
==================================================

Explicit negation must be preserved.

Negation indicators include:

not
no
without
never
did not
does not
do not
didn't
doesn't
but not
excluding

A negated action must have:

"negated": true

Example:

"A man did not enter the car."

Event:

{
    "event_id": 1,
    "action": "entering",
    "subject": "man",
    "object": "car",
    "attributes": [],
    "relations": [],
    "negated": true
}

==================================================
15. NEGATION SEPARATION
==================================================

Never hide a negated condition inside the attributes of a positive event.

Example:

"A man ate an apple without wearing a hat."

Correct:

Event 1:
man ate apple
negated = false

Event 2:
man wearing hat
negated = true

Negation:

{
    "event_id": 2,
    "trigger": "without"
}

Do NOT put "not wearing hat" inside event 1 attributes.

==================================================
16. EXPLICIT INFORMATION ONLY
==================================================

Only extract information explicitly present in the query.

Do NOT infer:

- gender from clothing
- color from context
- object identity from assumptions
- actions that are not stated
- locations that are not stated
- relationships that are not stated
- attributes that are not stated

==================================================
17. ENTITY UNIQUENESS
==================================================

Each explicitly mentioned entity should appear only once in "entities".

If the same entity is mentioned multiple times, reuse the same entity.

Do not create duplicate entities.

Example:

"A man entered a car. The man left the car."

Entities:

man
car

NOT:

man
car
man
car

==================================================
18. ENTITY TYPE CONSISTENCY
==================================================

The same entity must always keep the same type.

Do not assign multiple types to the same entity.

Example:

"car"

must remain:

{
    "type": "vehicle",
    "name": "car"
}

Do not later classify it as an object.

==================================================
19. ATTRIBUTE CONSISTENCY
==================================================

If an entity is described with attributes, preserve those attributes consistently.

Example:

"A red car was parked. The car was entered by a man."

The car remains:

name = car
attributes = ["red"]

==================================================
20. LOCATIONS
==================================================

Explicitly mentioned locations must appear as entities.

Example:

"A man entered a shop."

Entities:

man
shop

==================================================
21. NO EMPTY ENTITY NAMES
==================================================

Never create:

"name": ""

Every entity must have a valid name.

==================================================
22. NO INVENTION
==================================================

Never add information that does not exist in the query.

Do not create additional entities, attributes, events, relationships, timestamps, or temporal relationships.

==================================================
23. NO TIMESTAMPS
==================================================

Do not identify timestamps.

Do not estimate timestamps.

==================================================
24. NO VISUAL RETRIEVAL
==================================================

Only understand and structure the query.

Do not perform visual retrieval.

==================================================
25. JSON FORMAT
==================================================

Return ONLY valid JSON.

No markdown.

No explanations.

No comments.

No text outside the JSON object.

The output MUST follow this structure:

{
    "original_query": "...",
    "entities": [
        {
            "type": "person",
            "name": "...",
            "attributes": []
        },
        {
            "type": "object",
            "name": "...",
            "attributes": []
        },
        {
            "type": "vehicle",
            "name": "...",
            "attributes": []
        },
        {
            "type": "clothing",
            "name": "...",
            "attributes": []
        },
        {
            "type": "location",
            "name": "...",
            "attributes": []
        }
    ],
    "events": [
        {
            "event_id": 1,
            "action": "...",
            "subject": "...",
            "object": "...",
            "attributes": [],
            "relations": [],
            "negated": false
        }
    ],
    "temporal_relations": [
        {
            "relation": "before",
            "event1": 1,
            "event2": 2
        }
    ],
    "negations": [
        {
            "event_id": 2,
            "trigger": "not"
        }
    ]
}

==================================================
26. FINAL CHECK BEFORE OUTPUT
==================================================

Before returning the JSON, verify:

1. Every explicitly mentioned entity exists in "entities".
2. Every entity appears only once.
3. Entity names contain the core noun, not its adjectives.
4. Every explicit adjective is stored in "attributes".
5. "red top hat" becomes:
   name = "hat"
   attributes = ["red", "top"]
6. "black leather jacket" becomes:
   name = "jacket"
   attributes = ["black", "leather"]
7. Vehicles use type "vehicle".
8. Clothing uses type "clothing".
9. Distinct actions are separate events.
10. Shared subjects remain consistent.
11. Explicit temporal words create temporal relations.
12. Negated conditions have "negated": true.
13. Every negated event is recorded in "negations".
14. No unstated information is added.
15. The result is valid JSON.
16. Return ONLY the JSON object.

Return ONLY the JSON object.
"""

In [ ]:
def save_structured_query(data, filename="structured_query.json"):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            indent=4,
            ensure_ascii=False
        )

    print(f"Saved to: {filename}")

In [ ]:
def understand_query(query):
    prompt = PROMPT_FINAL + "\n\nUser Query:\n" + query

    response = ask_qwen(
        prompt,
        max_new_tokens=500
    )

    result = parse_json_response(response)

    result = verify_entities(
        query,
        result
    )

    result = verify_temporal_relations(
        query,
        result
    )

    verification = verify_negation(
        query,
        result
    )

    final_result = {
        "original_query": query,
        "structured_query": result,
        "negation_verification": verification
    }

    save_structured_query(final_result)

    return final_result

In [ ]:
query = input("Enter video search query: ")

structured_query = understand_query(query)

print(json.dumps(structured_query, indent=4))

Enter video search query:  A woman wearing a red dress picked up a black bag, walked toward a blue car, but did not get inside it. Later, a man in a gray jacket approached the car and drove away.


Saved to: structured_query.json
{
    "original_query": "A woman wearing a red dress picked up a black bag, walked toward a blue car, but did not get inside it. Later, a man in a gray jacket approached the car and drove away.",
    "structured_query": {
        "original_query": "A woman wearing a red dress picked up a black bag, walked toward a blue car, but did not get inside it. Later, a man in a gray jacket approached the car and drove away.",
        "entities": [
            {
                "type": "person",
                "name": "woman"
            },
            {
                "type": "person",
                "name": "man"
            },
            {
                "type": "clothing",
                "name": "red dress"
            },
            {
                "type": "clothing",
                "name": "gray jacket"
            },
            {
                "type": "object",
                "name": "black bag"
            },
            {
                "type